In [110]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


ImportError: Could not find the DLL(s) 'msvcp140_1.dll'. TensorFlow requires that these DLLs be installed in a directory that is named in your %PATH% environment variable. You may install these DLLs by downloading "Microsoft C++ Redistributable for Visual Studio 2015, 2017 and 2019" for your platform from this URL: https://support.microsoft.com/help/2977003/the-latest-supported-visual-c-downloads

In [ ]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "ORP"] # 6 entradas
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

N_COMPONENTS = 4

In [ ]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}"]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

# RNA

In [ ]:
from sklearn.neighbors import NearestNeighbors

def compute_d_max(X_orig, k=5, percentile=75):
    nn = NearestNeighbors(n_neighbors=k+1)  # +1 para excluir o próprio ponto
    nn.fit(X_orig)

    dist, _ = nn.kneighbors(X_orig)
    dist = dist[:, 1:]  # remove distância zero

    mean_dist = dist.mean(axis=1)
    return np.percentile(mean_dist, percentile)


def SelectRepresentativeVirtuals(
    X_orig, y_orig,
    X_virt, y_virt,
    k=5,
    alpha=1.0,
):
    d_max = compute_d_max(X_orig)
    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(X_orig)

    dist, idx = nn.kneighbors(X_virt)

    y_local_mean = np.array([y_orig[i].mean() for i in idx])
    y_local_std  = np.array([y_orig[i].std()  for i in idx])

    # evita std zero
    y_local_std[y_local_std == 0] = 1e-6

    mask_trend = np.abs(y_virt - y_local_mean) <= alpha * y_local_std

    if d_max is not None:
        mask_dist = dist.mean(axis=1) <= d_max
        mask = mask_trend & mask_dist
    else:
        mask = mask_trend

    return X_virt[mask], y_virt[mask]


def PlotVirtualSelection(
    X_orig, y_orig,
    X_virt, y_virt,
    X_virt_sel, y_virt_sel,
    feature_names,
    target,
    i,
    base_path="./Dados/VirtualData",
    title_prefix="Filtro por Tendência Local"
):
    """
    Gera subplots por feature usando índice de amostras no eixo x
    e salva o resultado em PDF.
    """

    n_features = X_orig.shape[1]

    # Caminho de saída
    out_dir = f"{base_path}/P{i+1}/FilterResults"
    os.makedirs(out_dir, exist_ok=True)

    out_file = f"{out_dir}/Filtered_{target}.pdf"

    # Identifica virtuais rejeitadas
    sel_idx = set(map(tuple, X_virt_sel))
    mask_sel = np.array([tuple(x) in sel_idx for x in X_virt])

    X_virt_rej = X_virt[~mask_sel]
    y_virt_rej = y_virt[~mask_sel]

    fig, axes = plt.subplots(
        n_features, 1,
        figsize=(10, 3 * n_features),
        sharey=True
    )

    if n_features == 1:
        axes = [axes]

    for j, ax in enumerate(axes):

        # Índices = quantidade de amostras
        idx_orig = np.arange(len(y_orig))
        idx_sel  = np.arange(len(y_virt_sel))
        idx_rej  = np.arange(len(y_virt_rej))

        # Dados reais
        ax.plot(
            idx_orig,
            y_orig,
            "o-", label="Originais"
        )

        # Virtuais selecionadas
        ax.plot(
            idx_sel,
            y_virt_sel,
            "o--", label="Virtuais selecionadas"
        )

        # Virtuais rejeitadas
        ax.scatter(
            idx_rej,
            y_virt_rej,
            alpha=0.35, label="Virtuais rejeitadas"
        )

        ax.set_xlabel(f"Índice da amostra ({feature_names[j]})")
        ax.grid(True)

        if j == 0:
            ax.legend(loc="best")

    fig.suptitle(f"{title_prefix} – Target: {target}", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    # Salva como PDF
    fig.savefig(out_file, format="pdf")
    plt.close(fig)

    print(f"✔ Figura salva em: {out_file}")


In [ ]:
def PrepareData(VirtualDataset, OriginalDataset, target, i):
    X_orig = OriginalDataset[PREDICTORS].values
    Y_orig = OriginalDataset[target].values
    
    Xv = VirtualDataset[PREDICTORS].values
    Yv = VirtualDataset[target].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_orig, Y_orig, test_size=0.2, random_state=42
    )
    
    # Xv_sel, yv_sel = SelectRepresentativeVirtuals(X_orig, Y_orig, Xv, Yv)
    
    # PlotVirtualSelection(
    # X_orig, Y_orig,
    # X_virt, Y_virt,
    # Xv_sel, yv_sel,
    # feature_names=PREDICTORS,
    # target=target,
    # i=i,
    # )
    
    # CONCATENAÇÃO CORRETA
    X_train = np.concatenate((X_train, Xv), axis=0)
    Y_train = np.concatenate((Y_train, Yv), axis=0)

    x_train = SCALER.fit_transform(X_train)
    x_test  = SCALER.transform(X_test)
    
    return x_train, x_test, Y_train, Y_test

    
def PrintDim(x, y):
    print(f"Dimensão da entrada: {np.shape(x)}")
    print(f"Dimensão da saida: {np.shape(y)}")

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

def TrainANN(
    x_train, y_train,
    x_test, y_test,
    n_hidden=[],
    lr=1e-3,
    l2_reg=1e-4,
    epochs=500,
    batch_size=8,
    verbose=0
):
    n_inputs = x_train.shape[1]

    # ======================
    # Modelo
    # ======================
    model = Sequential()

    model.add(
        Dense(
            n_hidden[0],
            activation="relu",
            kernel_regularizer=l2(l2_reg),
            input_shape=(n_inputs,)
        )
    )

    for units in n_hidden[1:]:
        model.add(
            Dense(
                units,
                activation="relu",
                kernel_regularizer=l2(l2_reg)
            )
        )

    model.add(Dense(1, activation="linear"))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True
    )

    # ======================
    # Treinamento
    # ======================
    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=verbose
    )

    # ======================
    # Predições (NORMALIZADAS)
    # ======================
    y_train_pred_norm = model.predict(x_train, verbose=0)
    y_test_pred_norm  = model.predict(x_test,  verbose=0)

    # ======================
    # DESNORMALIZAÇÃO
    # ======================
    y_train_real = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_test_real  = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()

    y_train_pred = OUT_SCALER.inverse_transform(y_train_pred_norm).ravel()
    y_test_pred  = OUT_SCALER.inverse_transform(y_test_pred_norm).ravel()

    # ======================
    # MÉTRICAS NO ESPAÇO FÍSICO
    # ======================
    metrics = {
        "mse_train": mean_squared_error(y_train_real, y_train_pred),
        "mse_test":  mean_squared_error(y_test_real,  y_test_pred),
        "r2_train":  r2_score(y_train_real, y_train_pred),
        "r2_test":   r2_score(y_test_real,  y_test_pred)
    }

    return model, history, metrics


In [ ]:
neurons = [1, 2, 4, 6, 8, 10, 14, 16, 18, 20]
all_metrics = []

for i, Dataset in enumerate(Datasets):
    os.makedirs(f"./Dados/VirtualData/P{i+1}/FilterResults/", exist_ok=True)

    for j, target in enumerate(TARGETS):
        
        print(f" → {target}")
        output_dir = f"./Dados/VirtualData/P{i+1}"
        vs_filename = os.path.join(output_dir, f"virtual_samples_{target}.xlsx")
        VirtualDataset = pd.read_excel(vs_filename, sheet_name="orig-vs")
        x_train, x_test, y_train, y_test = PrepareData(VirtualDataset, Dataset, target, i)
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        PrintDim(x_train, y_train)
        PrintDim(x_test, y_test)
        
        for neuron in neurons:

            model, history, metrics = TrainANN(
                x_train, y_train,
                x_test, y_test,
                neurons=neuron
            )

            # Apenas adiciona metadados
            metrics.update({
                "P": i + 1,
                "target": target,
                "neurons": neuron
            })

            all_metrics.append(metrics)
            break
     
        
        break
    break

 → Fe
Dimensão da entrada: (104, 6)
Dimensão da saida: (104,)
Dimensão da entrada: (6, 6)
Dimensão da saida: (6,)
